## **Breast Cancer Project**

---

### **Raw Data File Preprocessing and Formatting**

This notebook implements the main pipeline for breast cancer classification using RNA-Seq data. It includes data loading and preprocessing, statistical feature selection (ANOVA F-test), dimensionality reduction (PCA), and classification using a Random Forest model.

---

### **Setup and Configuration**

This step sets up the required libraries and defines the folder structure used in this notebook. The data is organized into four main folders under `data/`:
- `raw/`: Raw downloaded files (e.g., GTEx `.gct`, GDC `.tsv`).
- `initial/`: Processed raw files (e.g., merged and transposed).
- `interim/`: Feature-selected or partially transformed files.
- `processed/`: Final model-ready inputs (e.g., PCA-reduced features).

In [1]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
import shutil
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.metrics import roc_curve, auc

In [2]:
# Define data folder structure
DATA_DIR = "data"
RAW_GDC_DIR = os.path.join(DATA_DIR, "raw_gdc_data")
RAW_GTX_DIR = os.path.join(DATA_DIR, "raw_gtx_data")

INITIAL_DIR = os.path.join(DATA_DIR, "initial")
INTERIM_DIR = os.path.join(DATA_DIR, "interim")

In [3]:
# Create directories if they don't exist
os.makedirs(INITIAL_DIR, exist_ok=True)
os.makedirs(INTERIM_DIR, exist_ok=True)

---

### **Step 1: Initial Process of GDC Raw Sample Files (Cancer Data)**

This block processes RNA-Seq files downloaded from the GDC portal. Each `.tsv` file contains gene expression data for a single cancer sample. From each file, the column `tpm_unstranded` is extracted, indexed by `gene_id`.

These per-sample files are merged into a single matrix where:
- **Rows** = genes,
- **Columns** = samples,
- **Values** = TPM expression values.

This process results in a file `gdc_data.csv` saved to the `data/initial/` folder. 

It is computationally expensive, so it is designed to be skipped once completed unless the raw data changes.

---

##### **Step 1.1: Merge All Sample Files**
Merging separate cancer sample files and Building a complete dataset file

In [ ]:
# Define paths for GDC sample sheet and data files
gdc_sample_sheet_path = os.path.join(RAW_GDC_DIR, 'gdc_sample_sheet.tsv')
gdc_merged_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

sample_sheet_file_handler = pd.read_csv(gdc_sample_sheet_path, sep='\t')

# Initialize an empty DataFrame to store the combined data
gdc_full_df = pd.DataFrame()

sample_counter = 1
# Iterate through each file listed in the sample sheet
for index, row in sample_sheet_file_handler.iterrows():
    folder_id = row['File ID']
    file_name = row['File Name']
    
    # Construct the full file path
    sample_file_path = os.path.join(RAW_GDC_DIR, folder_id, file_name)
    
    # Read the TSV file
    try:
        sample_data = pd.read_csv(sample_file_path, sep='\t', skiprows=[0, 2, 3, 4, 5])
        
        # Extract the relevant columns ('gene_id' and 'TPM' or equivalent)
        relevant_data = sample_data[['gene_id', 'tpm_unstranded']]
        
        # Rename the columns to match the GTex format
        sample_number = 'c_' + str(sample_counter).zfill(4)
        relevant_data.columns = ['gene_id', sample_number]  # Use folder_id as the sample name
        
        # Merge with the combined data
        if gdc_full_df.empty:
            gdc_full_df = relevant_data
        else:
            gdc_full_df = pd.merge(gdc_full_df, relevant_data, on='gene_id', how='outer')

        sample_counter += 1
            
    except Exception as e:
        print(f"Error processing file {sample_file_path}: {e}")

gdc_full_df.to_csv(gdc_merged_file_path, index=False)

In [ ]:
# Load and display the first few rows of the merged GDC dataset
gdc_merged_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

gdc_merged_file_df = pd.read_csv(gdc_merged_file_path, index_col=0)
print("Shape of initial GDC Data Set", gdc_merged_file_df.shape)
gdc_merged_file_df.head(5)

##### **Step 1.2: Split into Model Training Data and Unseen Testing Data**
Building a model training dataset of `train_num` samples and an unseen dataset of `unseen_num` samples for testing fo deploying model.

Default Values of GDC tcga data for Breast Cancer: 
- Total number of samples: `total_num = 1231`
- Model Training `train_num= 1000`
- Unseen Testing Data `unseen_num = 231`

In [4]:
cancer_data_number_of_samples_for_model_builiding = 1000    # You can choose the number of data samples for model building
                                                            # The rest of data will be kept separately as Unseen data for testing

In [5]:
# Define path
gdc_data_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

gdc_full_data_few_rows_df = pd.read_csv(gdc_data_file_path, nrows=5)
columns = gdc_full_data_few_rows_df.columns.tolist()

gene_info_columns = columns[0:1]
sample_columns = columns[1:]

# Split data into data for model building and training and unseen data for testing
gdc_total_number_of_samples = len(sample_columns)         # bc tcga data = 1231
gdc_train_num = cancer_data_number_of_samples_for_model_builiding
gdc_unseen_num = gdc_total_number_of_samples - gdc_train_num

# Randomly select 1000 sample columns from the dataset as traiing data
gdc_chosen_columns_for_training = np.random.choice(sample_columns, gdc_train_num, replace=False).tolist()
gdc_final_columns_for_training = gene_info_columns + gdc_chosen_columns_for_training

# Get the remaining columns as testing data
gdc_chosen_columns_for_testing = [col for col in sample_columns if col not in gdc_chosen_columns_for_training]
gdc_final_columns_for_testing = gene_info_columns + gdc_chosen_columns_for_testing

# Load the dataset again but only with the selected columns
gdc_train_df = pd.read_csv(gdc_data_file_path, usecols=gdc_final_columns_for_training)
gdc_unseen_df = pd.read_csv(gdc_data_file_path, usecols=gdc_final_columns_for_testing)

# Save the training and testing data to new files
gdc_training_file_path = os.path.join(INITIAL_DIR, 'gdc_data_training.csv')
gdc_unseen_file_path = os.path.join(INITIAL_DIR, 'gdc_data_testing.csv')

gdc_train_df.to_csv(gdc_training_file_path, index=False)
gdc_unseen_df.to_csv(gdc_unseen_file_path, index=False)

In [6]:
print("Shape of initial GDC Model Building Data Set", gdc_train_df.shape)
gdc_train_df.head(5)

Shape of initial GDC Model Building Data Set (60660, 1001)


,gene_id,c_0001,c_0002,c_0005,c_0006,c_0007,c_0008,c_0009,c_0010,c_0012,...,c_1219,c_1221,c_1223,c_1224,c_1225,c_1226,c_1228,c_1229,c_1230,c_1231
0,ENSG00000000003.15,49.6341,12.0296,18.0535,10.6064,24.0318,63.9724,75.6881,11.2357,42.7436,...,16.2896,52.3403,16.6648,30.1448,26.2866,37.7670,71.7311,5.3684,113.1190,11.5596
1,ENSG00000000005.6,9.3826,0.3785,0.3523,0.6661,1.3877,0.3128,0.3828,0.0000,10.1675,...,0.7410,0.2608,0.1169,0.0567,8.9343,0.0000,0.6878,0.3944,0.4400,0.0424
2,ENSG00000000419.13,115.1737,134.9047,71.4260,112.2490,52.8140,125.0148,111.9771,66.6285,139.8421,...,87.3807,107.1986,85.6008,126.5208,63.2222,165.8215,115.8308,129.9716,82.5384,283.6449
3,ENSG00000000457.14,20.1202,15.5758,10.7816,3.3119,21.4261,15.2346,8.9836,3.8821,21.9184,...,9.2057,16.1887,12.5955,29.8414,11.5421,11.6838,16.0663,27.4784,19.6494,17.0290
4,ENSG00000000460.17,6.1859,4.3777,5.0191,3.8493,4.8368,9.3567,15.2681,2.3710,24.4534,...,1.2480,6.7480,4.9241,12.4256,4.6340,7.6037,4.8100,12.2774,8.9295,8.8565


In [7]:
print("Shape of initial GDC Testing Data Set", gdc_unseen_df.shape)
gdc_unseen_df.head(5)

Shape of initial GDC Testing Data Set (60660, 232)


,gene_id,c_0003,c_0004,c_0011,c_0017,c_0020,c_0021,c_0028,c_0033,c_0045,...,c_1185,c_1192,c_1202,c_1205,c_1209,c_1213,c_1215,c_1220,c_1222,c_1227
0,ENSG00000000003.15,90.4249,26.5679,13.0708,2.3364,66.0644,64.8954,55.0186,85.1012,31.8464,...,49.2199,11.4142,68.1033,72.3741,51.8718,8.1884,37.6335,84.4328,27.3544,85.6117
1,ENSG00000000005.6,0.0000,4.0213,0.1461,0.1670,8.9647,1.3858,1.2034,3.6590,2.3704,...,0.5422,0.7400,0.1034,1.3976,0.1706,0.0000,0.4165,12.1832,0.0979,0.3046
2,ENSG00000000419.13,110.8229,86.0077,156.3535,110.0601,62.2906,169.6122,143.1775,106.7875,61.6094,...,148.1766,117.2234,126.2405,96.3279,170.4536,72.5227,178.1384,111.7066,82.4554,148.1734
3,ENSG00000000457.14,31.8373,19.7490,11.0466,18.0468,14.5792,13.7604,18.0108,12.3342,18.4199,...,11.7568,13.5948,17.9539,13.2372,11.6550,12.6702,10.5387,22.8129,13.8403,12.4748
4,ENSG00000000460.17,15.4869,16.9112,7.1264,9.8942,2.6520,9.9658,20.9883,3.9606,9.2477,...,13.4292,6.6965,5.1302,7.3237,4.1838,4.2205,5.9564,4.6271,4.8875,7.6431


---

### **Step 2: Initial Process of GTex Raw Sample Files**

This step processes transcriptomic data from GTEx. The original file is in `.gct` format and includes expression levels for thousands of genes across thousands of samples.

This step includes:
1. Conversion of `.gct` to `.csv`, skipping initial metadata rows.
2. Random selection of columns (healthy samples) to build our hralthy data set
3. Saving the final dataset as `gtx_data.csv` in `data/initial/`.

Like GDC, this step is time-consuming and should be skipped in repeated runs unless data needs to be refreshed.

---

In [ ]:
# Define paths
gtx_full_gct_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.gct')
gtx_full_csv_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.csv')

##### **Step 2.1: Convert Raw GCT to Raw CSV file**
This block defines the `read_gct()` function, which skips the top two header lines from the `.gct` file and loads the remaining expression matrix into a DataFrame. The `convert_gct_to_csv()` function wraps this logic and saves the result as a temporary `.csv` file.

In [ ]:
def read_gct(file_path):
    with open(file_path, 'r') as f:
        # Skip the first two header lines
        for _ in range(2):
            next(f)
        # Read the rest of the file into a pandas DataFrame
        df = pd.read_csv(f, sep='\t')
    return df

In [ ]:
def convert_gct_to_csv(gct_file, csv_file):
    df = read_gct(gct_file)
    df.to_csv(csv_file, index=False)

In [ ]:
convert_gct_to_csv(gtx_full_gct_file_path, gtx_full_csv_file_path)

##### **Step 2.2: Select Healthy Samples Randomly to Create Healthy Data**


In [8]:
gtx_number_of_samples_to_be_selected = 2231

In [9]:
gtx_full_csv_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.csv')
gtx_full_df = pd.read_csv(gtx_full_csv_file_path, nrows=5)
columns = gtx_full_df.columns.tolist()

# The first two columns are 'Name' and 'Description', we keep them and sample the rest
gene_info_columns = columns[0:2]
sample_columns = columns[2:]

# Randomly select sample columns from the dataset
chosen_samples_columns = np.random.choice(sample_columns, gtx_number_of_samples_to_be_selected, replace=False).tolist()
final_samples_columns = gene_info_columns + chosen_samples_columns

# Load the dataset again but only with the selected columns
gtx_selected_data_df = pd.read_csv(gtx_full_csv_file_path, usecols=final_samples_columns)
gtx_selected_data_df = gtx_selected_data_df.rename(columns={'Name': 'gene_id'})
gtx_selected_data_df = gtx_selected_data_df.drop('Description', axis=1)

# Changing the sample ids to h_xxxx format
num_samples = gtx_selected_data_df.shape[1] - 1
new_sample_names = [f"h_{i:04d}" for i in range(1, num_samples + 1)]
gtx_selected_data_df.columns = [gtx_selected_data_df.columns[0]] + new_sample_names

# Save the sampled data to a new CSV file
gtx_selected_samples_file_path = os.path.join(INITIAL_DIR, 'gtx_data.csv')
gtx_selected_data_df.to_csv(gtx_selected_samples_file_path, index=False)

In [10]:
# Load and display the first few rows of the GTex selected dataset
gtx_data_final_df = pd.read_csv(gtx_selected_samples_file_path, index_col=0)
print("Shape of initial GTex Data Set", gtx_data_final_df.shape)
gtx_data_final_df.head(5)

Shape of initial GTex Data Set (56200, 2231)


,h_0001,h_0002,h_0003,h_0004,h_0005,h_0006,h_0007,h_0008,h_0009,h_0010,...,h_2222,h_2223,h_2224,h_2225,h_2226,h_2227,h_2228,h_2229,h_2230,h_2231
gene_id,,,,,,,,,,,,,,,,,,,,,
ENSG00000223972.5,0.00000,0.03757,0.01936,0.000,0.01768,0.000,0.0000,0.0000,0.0000,0.000,...,0.02944,0.00000,0.03553,0.02295,0.000,0.0000,0.000,0.00000,0.00000,0.00000
ENSG00000227232.5,16.95000,0.92950,1.23500,5.766,1.19700,3.378,0.9979,2.7940,0.7417,3.739,...,6.89900,1.49800,3.60800,2.89800,2.916,4.4060,3.732,1.92400,4.14000,2.46600
ENSG00000278267.1,0.00000,0.00000,0.00000,0.000,0.00000,0.000,0.0000,0.6737,0.0000,0.000,...,0.00000,0.00000,0.00000,0.00000,0.000,0.0000,0.000,0.00000,0.00000,0.00000
ENSG00000243485.5,0.00000,0.00000,0.00000,0.000,0.00000,0.000,0.0000,0.0000,0.0000,0.000,...,0.00000,0.02767,0.00000,0.00000,0.000,0.0508,0.000,0.00000,0.05038,0.00000
ENSG00000237613.2,0.03904,0.00000,0.00000,0.000,0.02508,0.000,0.0000,0.0000,0.0000,0.000,...,0.00000,0.00000,0.00000,0.00000,0.000,0.0000,0.000,0.03437,0.03579,0.05597


##### **Step 2.3: Dividing Gtex data into Training and (Unseen) Tesing Data**


In [11]:
gtx_selected_samples_file_path_ = os.path.join(INITIAL_DIR, 'gtx_data.csv')
gtx_selected_samples_few_rows_df = pd.read_csv(gtx_selected_samples_file_path_, nrows=5)
gtx_columns = gtx_selected_samples_few_rows_df.columns.tolist()

# The first two columns are 'Name' and 'Description', we keep them and sample the rest
gtx_gene_info_columns = gtx_columns[0:1]
gtx_samples_columns = gtx_columns[1:]

# Split data into data for model building and training and unseen data for testing
gtx_total_num = len(gtx_samples_columns)
gtx_train_num = 1000
gtx_unseen_num = gtx_total_num - gtx_train_num      # The rest of samples will be used for testing as unseen data

# Randomly select x sample columns from the dataset as traiing data
gtx_chosen_columns_for_training = np.random.choice(gtx_samples_columns, gtx_train_num, replace=False).tolist()
gtx_final_columns_for_training = gtx_gene_info_columns + gtx_chosen_columns_for_training

# Select the rest of samples from the dataset as testing data
gtx_chosen_columns_for_testing = [col for col in gtx_samples_columns if col not in gtx_chosen_columns_for_training]
gtx_final_columns_for_testing = gtx_gene_info_columns + gtx_chosen_columns_for_testing

# Load the dataset again but only with the selected columns
gtx_train_df = pd.read_csv(gtx_selected_samples_file_path_, usecols=gtx_final_columns_for_training)
gtx_unseen_df = pd.read_csv(gtx_selected_samples_file_path_, usecols=gtx_final_columns_for_testing)

# Save the training and unseen testing data to new CSV files
gtx_training_file_path = os.path.join(INITIAL_DIR, 'gtx_data_training.csv')
gtx_testing_file_path = os.path.join(INITIAL_DIR, 'gtx_data_testing.csv')

gtx_train_df.to_csv(gtx_training_file_path, index=False)
gtx_unseen_df.to_csv(gtx_testing_file_path, index=False)

In [12]:
print("Shape of initial Gtex Training Data Set", gtx_train_df.shape)
gtx_train_df.head(5)

Shape of initial Gtex Training Data Set (56200, 1001)


,gene_id,h_0003,h_0004,h_0006,h_0007,h_0009,h_0010,h_0012,h_0016,h_0017,...,h_2219,h_2220,h_2223,h_2224,h_2225,h_2226,h_2227,h_2228,h_2230,h_2231
0,ENSG00000223972.5,0.01936,0.000,0.000,0.0000,0.0000,0.000,0.000,0.03048,0.000,...,0.02614,0.000,0.00000,0.03553,0.02295,0.000,0.0000,0.000,0.00000,0.00000
1,ENSG00000227232.5,1.23500,5.766,3.378,0.9979,0.7417,3.739,4.206,4.48400,8.299,...,2.92700,2.463,1.49800,3.60800,2.89800,2.916,4.4060,3.732,4.14000,2.46600
2,ENSG00000278267.1,0.00000,0.000,0.000,0.0000,0.0000,0.000,0.000,0.00000,0.000,...,0.00000,0.000,0.00000,0.00000,0.00000,0.000,0.0000,0.000,0.00000,0.00000
3,ENSG00000243485.5,0.00000,0.000,0.000,0.0000,0.0000,0.000,0.000,0.00000,0.000,...,0.00000,0.000,0.02767,0.00000,0.00000,0.000,0.0508,0.000,0.05038,0.00000
4,ENSG00000237613.2,0.00000,0.000,0.000,0.0000,0.0000,0.000,0.000,0.00000,0.000,...,0.00000,0.000,0.00000,0.00000,0.00000,0.000,0.0000,0.000,0.03579,0.05597


In [13]:
print("Shape of initial GTex Testing Data Set", gtx_unseen_df.shape)
gtx_unseen_df.head(5)

Shape of initial GTex Testing Data Set (56200, 1232)


,gene_id,h_0001,h_0002,h_0005,h_0008,h_0011,h_0013,h_0014,h_0015,h_0022,...,h_2209,h_2210,h_2212,h_2213,h_2215,h_2216,h_2218,h_2221,h_2222,h_2229
0,ENSG00000223972.5,0.00000,0.03757,0.01768,0.0000,0.00000,0.00000,0.05453,0.00000,0.000,...,0.00000,0.02608,0.00000,0.00000,0.02168,0.00000,0.00000,0.03807,0.02944,0.00000
1,ENSG00000227232.5,16.95000,0.92950,1.19700,2.7940,4.87500,1.24200,3.40800,12.30000,1.894,...,2.83000,2.95400,10.25000,5.86800,1.94800,5.80200,2.31400,8.42600,6.89900,1.92400
2,ENSG00000278267.1,0.00000,0.00000,0.00000,0.6737,0.00000,0.00000,0.00000,0.00000,0.000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3,ENSG00000243485.5,0.00000,0.00000,0.00000,0.0000,0.07628,0.02608,0.00000,0.07285,0.000,...,0.04616,0.00000,0.07308,0.00000,0.04328,0.05740,0.03735,0.07601,0.00000,0.00000
4,ENSG00000237613.2,0.03904,0.00000,0.02508,0.0000,0.00000,0.00000,0.00000,0.00000,0.000,...,0.00000,0.07399,0.00000,0.04151,0.00000,0.04078,0.00000,0.00000,0.00000,0.03437


---

### **Step 3: Create Final Data - Merge and Label Cancer and Non-Cancer**

---

##### **Step 3.1: Create Training Data (Combined Cancer and non-Cancer)**

This step merges the GTEx (non-cancer) and GDC (cancer) RNA-Seq datasets and prepares them for machine learning. The goal is to align the gene expression profiles from both sources, assign binary labels, and generate a unified dataset.

This step includes:
1. **Loads**:
   - `gtx_data.csv` and `gdc_data.csv` from `data/initial/`
2. **Aligns gene features** (columns) to keep only shared genes between GTEx and GDC.
3. **Transposes the matrices**:
   - Each **row** becomes a sample,
   - Each **column** is a gene (TPM value).
4. **Assigns labels**:
   - `0` for GTEx (healthy samples),
   - `1` for GDC (cancer samples).
5. **Combines** both datasets into a single feature matrix and a corresponding label vector.

##### Output files (in `data/interim/`):
- `preprocessed_data_features.csv`: Combined matrix of all samples with aligned gene features (samples × genes)
- `preprocessed_data_labels.csv`: Binary labels for each sample (0 = GTEx, 1 = GDC)

These files serve as input to the next steps: statistical feature selection and dimensionality reduction.


In [14]:
gtx_file_path = os.path.join(INITIAL_DIR, 'gtx_data_training.csv')
gdc_file_path = os.path.join(INITIAL_DIR, 'gdc_data_training.csv')

# Load the datasets with headers
gtx__df = pd.read_csv(gtx_file_path, header=0)
gdc__df = pd.read_csv(gdc_file_path, header=0)

# Ensure that the 'gene_id' column is the index for both datasets
gtx__df.set_index('gene_id', inplace=True)
gdc__df.set_index('gene_id', inplace=True)

# Add labels row: 0 for GTEx (healthy) and 1 for GDC (cancer)
gtx_labels = pd.DataFrame([0] * gtx__df.shape[1], index=gtx__df.columns, columns=['label']).transpose()
gdc_labels = pd.DataFrame([1] * gdc__df.shape[1], index=gdc__df.columns, columns=['label']).transpose()

# Concatenate labels and data
gtx__df = pd.concat([gtx_labels, gtx__df])
gdc__df = pd.concat([gdc_labels, gdc__df])

# Find common genes (rows)
common_genes = gtx__df.index.intersection(gdc__df.index)

# Filter both datasets to keep only the common genes
gtx_df_aligned = gtx__df.loc[common_genes]
gdc_df_aligned = gdc__df.loc[common_genes]

# Combine both datasets
combined_data = pd.concat([gtx_df_aligned, gdc_df_aligned], axis=1)

# Transpose the data to have samples as rows and genes as columns
combined_data = combined_data.transpose()

# Separate features and labels
labels = combined_data['label']
features = combined_data.drop(columns=['label'])

# Save preprocessed features and labels to CSV
output_file_prefix = os.path.join(INTERIM_DIR, 'training_data')
features.to_csv(output_file_prefix + '_features.csv', index=False)
labels.to_csv(output_file_prefix + '_labels.csv', index=False)

print("Data Merging Complete")
print("Shape of features:", features.shape)
print("Shape of labels:", labels.shape)

Data Merging Complete
Shape of features: (2000, 35117)
Shape of labels: (2000,)


##### **Step 3.2: Create Unseen Testing Data (Combined Cancer and non-Cancer)**


In [15]:
gtx_test_file_path = os.path.join(INITIAL_DIR, 'gtx_data_testing.csv')
gdc_test_file_path = os.path.join(INITIAL_DIR, 'gdc_data_testing.csv')

# Load the datasets with headers
gtx_test__df = pd.read_csv(gtx_test_file_path, header=0)
gdc_test__df = pd.read_csv(gdc_test_file_path, header=0)

# Ensure that the 'gene_id' column is the index for both datasets
gtx_test__df.set_index('gene_id', inplace=True)
gdc_test__df.set_index('gene_id', inplace=True)

# Add labels row: 0 for GTEx (healthy) and 1 for GDC (cancer)
gtx_test_labels = pd.DataFrame([0] * gtx_test__df.shape[1], index=gtx_test__df.columns, columns=['label']).transpose()
gdc_test_labels = pd.DataFrame([1] * gdc_test__df.shape[1], index=gdc_test__df.columns, columns=['label']).transpose()

# Concatenate labels and data
gtx_test__df = pd.concat([gtx_test_labels, gtx_test__df])
gdc_test__df = pd.concat([gdc_test_labels, gdc_test__df])

# Find common genes (rows)
common_genes = gtx_test__df.index.intersection(gdc_test__df.index)

# Filter both datasets to keep only the common genes
gtx_test_df_aligned = gtx_test__df.loc[common_genes]
gdc_test_df_aligned = gdc_test__df.loc[common_genes]

# Combine both datasets
combined_data_test = pd.concat([gtx_test_df_aligned, gdc_test_df_aligned], axis=1)

# Transpose the data to have samples as rows and genes as columns
combined_data_test = combined_data_test.transpose()

# Separate features and labels
labels_test = combined_data_test['label']
features_test = combined_data_test.drop(columns=['label'])

# Save preprocessed features and labels to CSV
output_file_prefix = os.path.join(INTERIM_DIR, 'testing_data')
features_test.to_csv(output_file_prefix + '_features.csv', index=False)
labels_test.to_csv(output_file_prefix + '_labels.csv', index=False)

print("Data Merging Complete")
print("Shape of features:", features_test.shape)
print("Shape of labels:", labels_test.shape)

Data Merging Complete
Shape of features: (1462, 35117)
Shape of labels: (1462,)
